<a href="https://colab.research.google.com/github/solankiSarvesh/HealthScribe-AI/blob/main/GenAI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes
!pip install -q faiss-cpu sentence-transformers
!pip install -q streamlit pyngrok
!pip install -q spacy rouge-score nltk
!pip install -q datasets trl
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('All dependencies installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed!


In [ ]:
from google.colab import files
import pandas as pd

print('📂 Upload your medical_reports.csv file...')
uploaded = files.upload()

df = pd.read_csv('medical_reports.csv')
print(f'✅ Loaded {len(df)} records')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

📂 Upload your medical_reports.csv file...


Saving medical_reports.csv to medical_reports (1).csv
✅ Loaded 1404 records
Columns: ['report', 'explanation']


,report,explanation
0,"Hemoglobin: 11.0 g/dL, WBC: 7935",Mildly low hemoglobin may indicate mild anemia.
1,"Hemoglobin: 7.5 g/dL, WBC: 4906",Low hemoglobin suggests anemia; further workup...
2,"Creatinine: 3.2 mg/dL, BUN: 31 mg/dL",Significantly elevated creatinine indicates mo...


In [ ]:
import re
import spacy
import sqlite3
import numpy as np

nlp = spacy.load('en_core_web_sm')

MEDICAL_PATTERNS = [
    r'\b(Hemoglobin|Hb|WBC|RBC|Platelets|Glucose|Creatinine|BP|'
    r'HbA1c|ALT|AST|TSH|Cholesterol|Sodium|Potassium|Urea)\b',
    r'\b\d+\.?\d*\s*(g/dL|mg/dL|mmol/L|U/L|mIU/L|%|mm Hg|cells/mcL)\b'
]

def clean_text(text):
    text = str(text)
    text = re.sub(r'[^\w\s\.\,\:\;\(\)\-\/\%]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'(\d)\s+(g/dL|mg/dL|mmol/L)', r'\1 \2', text)
    return text

def extract_entities(text):
    entities = []
    for pattern in MEDICAL_PATTERNS:
        matches = re.findall(pattern, text, re.IGNORECASE)
        entities.extend(matches)
    doc = nlp(text)
    spacy_ents = [(ent.text, ent.label_) for ent in doc.ents
                  if ent.label_ in ('QUANTITY', 'CARDINAL', 'PERCENT', 'PRODUCT')]
    entities.extend([e[0] for e in spacy_ents])
    return list(set(entities))

def flag_abnormal(text):
    flags = []
    rules = [
        (r'Hemoglobin.*?(\d+\.?\d*)\s*g/dL', lambda v: float(v) < 12, '⚠️ Low Hemoglobin (possible anemia)'),
        (r'WBC.*?(\d+)\s*cells', lambda v: int(v) > 11000, '⚠️ Elevated WBC (possible infection/inflammation)'),
        (r'Glucose.*?(\d+)\s*mg/dL', lambda v: float(v) > 126, '⚠️ High Glucose (possible diabetes risk)'),
        (r'Creatinine.*?(\d+\.?\d*)\s*mg/dL', lambda v: float(v) > 1.3, '⚠️ High Creatinine (possible kidney stress)'),
        (r'Cholesterol.*?(\d+)\s*mg/dL', lambda v: float(v) > 200, '⚠️ High Cholesterol (cardiovascular risk)'),
        (r'HbA1c.*?(\d+\.?\d*)\s*%', lambda v: float(v) > 6.5, '⚠️ Elevated HbA1c (long-term glucose concern)'),
    ]
    for pattern, condition, message in rules:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            try:
                if condition(match.group(1)):
                    flags.append(message)
            except:
                pass
    return flags

df['report_clean']      = df['report'].apply(clean_text)
df['explanation_clean'] = df['explanation'].apply(clean_text)
df['entities']          = df['report_clean'].apply(extract_entities)
df['abnormal_flags']    = df['report_clean'].apply(flag_abnormal)

print('✅ Preprocessing done!')
print(f'Sample entities: {df["entities"].iloc[0]}')
print(f'Sample flags:    {df["abnormal_flags"].iloc[0]}')

✅ Preprocessing done!
Sample entities: ['11.0', 'Hemoglobin', 'WBC', 'g/dL']
Sample flags:    ['⚠️ Low Hemoglobin (possible anemia)']


In [ ]:
import sqlite3, json

conn = sqlite3.connect('medical_reports.db')
c = conn.cursor()
c.execute('''
    CREATE TABLE IF NOT EXISTS reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_raw       TEXT,
        report_clean     TEXT,
        explanation      TEXT,
        entities         TEXT,
        abnormal_flags   TEXT,
        ai_summary       TEXT,
        ai_explanation   TEXT,
        created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')
conn.commit()

for _, row in df.iterrows():
    c.execute('''
        INSERT INTO reports (report_raw, report_clean, explanation, entities, abnormal_flags)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        row['report'],
        row['report_clean'],
        row['explanation_clean'],
        json.dumps(row['entities']),
        json.dumps(row['abnormal_flags'])
    ))
conn.commit()
conn.close()
print(f'✅ SQLite DB created with {len(df)} records → medical_reports.db')

✅ SQLite DB created with 1404 records → medical_reports.db


In [ ]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer

print('🔄 Loading sentence-transformers model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('🔄 Generating embeddings...')
embeddings = embedder.encode(df['report_clean'].tolist(),
                              batch_size=32, show_progress_bar=True)
embeddings = np.array(embeddings, dtype='float32')

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

faiss.write_index(index, 'faiss_medical.index')
with open('faiss_texts.pkl', 'wb') as f:
    pickle.dump({'reports': df['report_clean'].tolist(),
                 'explanations': df['explanation_clean'].tolist()}, f)

print(f'✅ FAISS index built with {index.ntotal} vectors (dim={dim})')

🔄 Loading sentence-transformers model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Generating embeddings...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

✅ FAISS index built with 1404 vectors (dim=384)


In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

# ── Gemma-2B-IT ───────────────────────────────────────────────────────────────
MODEL_NAME = 'google/gemma-2b-it'

print(f'🔄 Loading base model: {MODEL_NAME}')
print('⚠️  You must accept Gemma license at huggingface.co/google/gemma-2b-it')
print('    Then run: huggingface-cli login  (or set HF_TOKEN below)\n')

# ── HuggingFace login (required for Gemma) ────────────────────────────────────
from huggingface_hub import login
import os

HF_TOKEN = ""   # ← paste your token here: hf_xxxxxxxxxxxx
                 #   get it from huggingface.co/settings/tokens

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()      # interactive prompt

# ── 4-bit quantization config ─────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ── Load tokenizer ────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN if HF_TOKEN else True,
    trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

# ── Load base model ───────────────────────────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN if HF_TOKEN else True,
    trust_remote_code=True
)
base_model = prepare_model_for_kbit_training(base_model)
print('✅ Gemma-2B-IT loaded in 4-bit!\n')

# ── LoRA config — Gemma uses different attention module names ─────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',   # Gemma attention layers
        'gate_proj', 'up_proj', 'down_proj'         # Gemma MLP layers
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# ── Gemma chat format: <start_of_turn>user ... <end_of_turn><start_of_turn>model ──
def format_prompt(row):
    return (
        f"<start_of_turn>user\n"
        f"You are a medical report simplifier. "
        f"Explain this report in plain language for a patient:\n\n"
        f"{row['report_clean']}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{row['explanation_clean']}<end_of_turn>"
    )

df['text']  = df.apply(format_prompt, axis=1)
train_size  = int(0.85 * len(df))
train_df    = df['text'].iloc[:train_size].reset_index(drop=True)
hf_dataset  = Dataset.from_dict({'text': train_df.tolist()})

print(f'\n📋 Training samples : {len(hf_dataset)}')
print(f'📋 Eval samples     : {len(df) - train_size}')
print(f'\nSample prompt:\n{df["text"].iloc[0][:300]}\n')

# ── Training args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir='./gemma_medical',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_32bit',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args
)

print('🚀 Starting LoRA fine-tuning on Gemma-2B-IT...')
trainer.train()
model.save_pretrained('./gemma_medical_final')
tokenizer.save_pretrained('./gemma_medical_final')
print('✅ Fine-tuning complete! Model saved → ./gemma_medical_final')

🔄 Loading base model: google/gemma-2b-it
⚠️  You must accept Gemma license at huggingface.co/google/gemma-2b-it
    Then run: huggingface-cli login  (or set HF_TOKEN below)



Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

✅ Gemma-2B-IT loaded in 4-bit!

trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765

📋 Training samples : 1193
📋 Eval samples     : 211

Sample prompt:
<start_of_turn>user
You are a medical report simplifier. Explain this report in plain language for a patient:

Hemoglobin: 11.0 g/dL, WBC: 7935<end_of_turn>
<start_of_turn>model
Mildly low hemoglobin may indicate mild anemia.<end_of_turn>



Adding EOS to train dataset:   0%|          | 0/1193 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1193 [00:00<?, ? examples/s]

🚀 Starting LoRA fine-tuning on Gemma-2B-IT...


Step,Training Loss
10,5.815979
20,1.768974
30,1.041940
40,0.661330
50,0.585970
60,0.498381
70,0.395657
80,0.410233
90,0.382185
100,0.313740


✅ Fine-tuning complete! Model saved → ./gemma_medical_final


In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
)
from peft import PeftModel
import warnings, json, re
import numpy as np
import torch
warnings.filterwarnings('ignore')

scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method4

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ─────────────────────────────────────────────────────────────────────────────
# LOAD BASELINE — Raw gemma-2b (not instruction tuned)
# ─────────────────────────────────────────────────────────────────────────────
print('🔄 [1/2] Loading RAW baseline: google/gemma-2b ...')

raw_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2b', token=True)
raw_tokenizer.pad_token    = raw_tokenizer.eos_token
raw_tokenizer.padding_side = 'right'

raw_model = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2b',
    quantization_config=bnb_config,
    device_map='auto',
    token=True
)
raw_model.config.use_cache = True
if hasattr(raw_model, 'gradient_checkpointing_disable'):
    raw_model.gradient_checkpointing_disable()
raw_model.eval()
print('✅ Raw baseline loaded!\n')

# ─────────────────────────────────────────────────────────────────────────────
# RELOAD FINE-TUNED MODEL FRESH — fixes the repetition issue
# We reload base gemma-2b-it cleanly and attach LoRA weights fresh
# This avoids any leftover training state causing degenerate outputs
# ─────────────────────────────────────────────────────────────────────────────
print('🔄 [2/2] Reloading fine-tuned model fresh from saved weights...')

ft_tokenizer = AutoTokenizer.from_pretrained(
    "./gemma_medical_final",
    token=True
)
ft_tokenizer.pad_token    = ft_tokenizer.eos_token
ft_tokenizer.padding_side = 'right'

# Load clean base IT model
ft_base = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2b-it',
    quantization_config=bnb_config,
    device_map='auto',
    token=True
)
ft_base.config.use_cache = True
if hasattr(ft_base, 'gradient_checkpointing_disable'):
    ft_base.gradient_checkpointing_disable()

# Attach saved LoRA adapters
ft_model = PeftModel.from_pretrained(ft_base, './gemma_medical_final')
ft_model.config.use_cache = True
ft_model.eval()
print('✅ Fine-tuned model reloaded fresh!\n')

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT BUILDERS
# ─────────────────────────────────────────────────────────────────────────────

def build_baseline_prompt(report_text):
    """Plain completion — raw model has no instruction following"""
    return f"Medical report: {report_text.strip()}\nSummary:"

def build_finetuned_prompt(report_text):
    """Exact prompt used during LoRA training"""
    return (
        "<start_of_turn>user\n"
        "Explain the following medical report in simple plain English. "
        "Write 2-3 clear sentences. Do not repeat words.\n\n"
        f"{report_text.strip()}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

# ─────────────────────────────────────────────────────────────────────────────
# STOPPING CRITERIA
# ─────────────────────────────────────────────────────────────────────────────

def make_stopping_criteria(tok):
    end_id = tok.convert_tokens_to_ids('<end_of_turn>')
    eos_id = tok.eos_token_id

    class StopOnEnd(StoppingCriteria):
        def __call__(self, input_ids, scores, **kwargs):
            return input_ids[0][-1].item() in (end_id, eos_id)

    return StoppingCriteriaList([StopOnEnd()])

# ─────────────────────────────────────────────────────────────────────────────
# CLEAN OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

def clean_output(text):
    if not text:
        return ''

    # Remove Gemma special tags
    for tag in ['<end_of_turn>', '<start_of_turn>', 'model', 'user']:
        text = text.replace(tag, '')

    # Remove HTML/angle bracket junk
    text = re.sub(r'<[^>]{0,40}>', '', text)

    # Collapse repeated words: "Sever Sever sever" → "Severe"
    text = re.sub(r'\b(\w+)(\s+\1){2,}\b', r'\1', text, flags=re.IGNORECASE)

    # Collapse no-space stutter: "severseversever" → "sever"
    text = re.sub(r'\b(\w{4,}?)\1{2,}\b', r'\1', text, flags=re.IGNORECASE)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Deduplicate sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 5]
    seen, unique = [], []
    for s in sentences:
        key = re.sub(r'\s+', ' ', s.lower().strip('.!? '))
        if key not in seen:
            seen.append(key)
            unique.append(s)

    return ' '.join(unique[:3])

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE — greedy for fine-tuned (stable), sampling for baseline
# ─────────────────────────────────────────────────────────────────────────────

def generate(mdl, tok, prompt, is_raw=False, max_new=150):
    mdl.config.use_cache = True

    encoded = tok(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=420,
        add_special_tokens=True,
        padding=False
    )
    input_ids      = encoded['input_ids'].to(mdl.device)
    attention_mask = torch.ones_like(input_ids)
    input_len      = input_ids.shape[1]

    eos_id = tok.eos_token_id
    end_id = tok.eos_token_id if is_raw else tok.convert_tokens_to_ids('<end_of_turn>')

    if is_raw:
        # Baseline — sampling is fine, model will produce low quality anyway
        gen_kwargs = dict(
            input_ids            = input_ids,
            attention_mask       = attention_mask,
            max_new_tokens       = max_new,
            do_sample            = True,
            temperature          = 0.8,
            top_p                = 0.92,
            top_k                = 50,
            repetition_penalty   = 1.5,
            no_repeat_ngram_size = 3,
            pad_token_id         = eos_id,
            eos_token_id         = [eos_id, end_id],
            use_cache            = True
        )
    else:
        # Fine-tuned — use GREEDY decoding for stable clean output
        # no_repeat_ngram_size blocks the repetition loop entirely
        gen_kwargs = dict(
            input_ids            = input_ids,
            attention_mask       = attention_mask,
            max_new_tokens       = max_new,
            do_sample            = False,          # ← greedy, no randomness
            num_beams            = 4,              # ← beam search for quality
            repetition_penalty   = 2.0,
            no_repeat_ngram_size = 4,
            early_stopping       = True,
            pad_token_id         = eos_id,
            eos_token_id         = [eos_id, end_id],
            stopping_criteria    = make_stopping_criteria(tok),
            use_cache            = True
        )

    with torch.no_grad():
        output_ids = mdl.generate(**gen_kwargs)

    new_tokens = output_ids[0][input_len:]
    decoded    = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return clean_output(decoded)

# ─────────────────────────────────────────────────────────────────────────────
# QUICK SANITY CHECK before full eval
# ─────────────────────────────────────────────────────────────────────────────
print('🧪 Sanity check on 2 samples (fine-tuned model):\n' + '─'*55)
for i in range(min(3, len(df))):
    rep = df['report_clean'].iloc[i]
    ref = df['explanation_clean'].iloc[i]
    gen = generate(ft_model, ft_tokenizer, build_finetuned_prompt(rep), is_raw=False)
    print(f'Sample {i+1}')
    print(f'  REPORT : {rep[:80]}')
    print(f'  REF    : {ref[:80]}')
    print(f'  GEN    : {gen[:120]}')
    print(f'  Words  : {len(gen.split())}')
    print('─'*55)

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

eval_df = df.iloc[train_size:].reset_index(drop=True).head(20)

def evaluate_model(mdl, tok, label, prompt_fn, is_raw=False):
    r1s, r2s, rLs, bleus = [], [], [], []
    skipped = 0

    for i, (_, row) in enumerate(eval_df.iterrows()):
        pred = generate(mdl, tok, build_finetuned_prompt(row['report_clean'])
                        if not is_raw else prompt_fn(row['report_clean']),
                        is_raw=is_raw)
        ref  = row['explanation_clean']

        if len(pred.split()) < 4:
            skipped += 1
            continue

        if i < 2:
            print(f'  Sample {i+1}:')
            print(f'  REF  : {ref[:90]}')
            print(f'  PRED : {pred[:90]}\n')

        s = scorer.score(ref, pred)
        r1s.append(s['rouge1'].fmeasure)
        r2s.append(s['rouge2'].fmeasure)
        rLs.append(s['rougeL'].fmeasure)
        bleus.append(sentence_bleu(
            [ref.split()], pred.split(), smoothing_function=smoother
        ))

    if not r1s:
        print(f'❌ All outputs invalid for: {label}')
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'bleu': 0.0}

    result = {
        'rouge1': float(np.mean(r1s)),
        'rouge2': float(np.mean(r2s)),
        'rougeL': float(np.mean(rLs)),
        'bleu':   float(np.mean(bleus))
    }

    print(f'\n{"═"*52}')
    print(f'  {label}')
    print(f'{"═"*52}')
    print(f'  ROUGE-1 : {result["rouge1"]:.4f}')
    print(f'  ROUGE-2 : {result["rouge2"]:.4f}')
    print(f'  ROUGE-L : {result["rougeL"]:.4f}')
    print(f'  BLEU    : {result["bleu"]:.4f}')
    print(f'  Valid   : {len(r1s)}/20  (skipped {skipped})')
    print(f'{"═"*52}\n')
    return result

# ─────────────────────────────────────────────────────────────────────────────
# RUN BOTH EVALUATIONS
# ─────────────────────────────────────────────────────────────────────────────

print('\n' + '='*52)
print('  EVALUATION: Baseline vs LoRA Fine-Tuned')
print('='*52)

print('\n📊 [1/2] Baseline — Raw Gemma-2B...')
scores_baseline = evaluate_model(
    raw_model, raw_tokenizer,
    'Baseline — Raw Gemma-2B',
    build_baseline_prompt,
    is_raw=True
)

print('\n📊 [2/2] Final — LoRA Fine-Tuned Gemma-2B-IT...')
scores_finetuned = evaluate_model(
    ft_model, ft_tokenizer,
    'Final — LoRA Fine-Tuned Gemma-2B-IT',
    build_finetuned_prompt,
    is_raw=False
)

# ── Save ──────────────────────────────────────────────────────────────────────
with open('eval_scores.json', 'w') as f:
    json.dump({'baseline': scores_baseline, 'finetuned': scores_finetuned},
              f, indent=2)
print('✅ eval_scores.json saved!')

try:
    import shutil
    shutil.copy('eval_scores.json', DRIVE_SCORES)
    print('✅ Synced to Drive')
except:
    pass

# ── Final table ───────────────────────────────────────────────────────────────
print('\n' + '═'*58)
print(f'  {"Metric":<10} {"Baseline (Raw)":>16} {"LoRA Fine-Tuned":>16} {"Gain":>10}')
print('─'*58)
for m, lbl in [('rouge1','ROUGE-1'),
               ('rouge2','ROUGE-2'),
               ('rougeL','ROUGE-L'),
               ('bleu',  'BLEU   ')]:
    b    = scores_baseline[m]
    ft   = scores_finetuned[m]
    gain = (ft - b) * 100
    arrow = '📈' if ft > b else '📉'
    print(f'  {lbl:<10} {b:>16.4f} {ft:>16.4f} {arrow} {gain:>+7.1f}%')
print('═'*58)

🔄 [1/2] Loading RAW baseline: google/gemma-2b ...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

✅ Raw baseline loaded!

🔄 [2/2] Reloading fine-tuned model fresh from saved weights...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

✅ Fine-tuned model reloaded fresh!

🧪 Sanity check on 3 samples (fine-tuned model):
───────────────────────────────────────────────────────
Sample 1
  REPORT : Hemoglobin: 11.0 g/dL, WBC: 7935
  REF    : Mildly low hemoglobin may indicate mild anemia.
  GEN    : Mildly low hemoglobin may indicate mild anemia.
  Words  : 7
───────────────────────────────────────────────────────
Sample 2
  REPORT : Hemoglobin: 7.5 g/dL, WBC: 4906
  REF    : Low hemoglobin suggests anemia; further workup needed.
  GEN    : Low hemoglobin suggests anemia; further workup needed.
  Words  : 7
───────────────────────────────────────────────────────
Sample 3
  REPORT : Creatinine: 3.2 mg/dL, BUN: 31 mg/dL
  REF    : Significantly elevated creatinine indicates moderate to severe kidney disease.
  GEN    : Significantly elevated creatinine indicates moderate to severe kidney disease.
  Words  : 9
───────────────────────────────────────────────────────

  EVALUATION: Baseline vs LoRA Fine-Tuned

📊 [1/2] Baseline 

# Save and Reload

In [ ]:
# =========================
# SAVE FULL WORK TO DRIVE (FINAL VERSION)
# =========================

# Install required packages (safe to rerun)
!pip install -q transformers peft accelerate bitsandbytes
!pip install -q faiss-cpu sentence-transformers
!pip install -q streamlit pyngrok
!pip install -q spacy rouge-score nltk datasets trl joblib sentencepiece

import os
import json
import pickle
import shutil
import sqlite3
import joblib
import torch
import pandas as pd
import numpy as np
import faiss
import nltk
import spacy

from google.colab import drive
drive.mount('/content/drive')

# =========================
# CREATE SAVE DIRECTORY
# =========================
SAVE_DIR = "/content/drive/MyDrive/genai_project_full_backup"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving everything to:", SAVE_DIR)

# =========================
# SAVE DATAFRAME
# =========================
if 'df' in globals():
    df.to_csv(f"{SAVE_DIR}/medical_reports_processed.csv", index=False)
    print("DataFrame saved")

# =========================
# SAVE SQLITE DATABASE
# =========================
if os.path.exists("medical_reports.db"):
    shutil.copy("medical_reports.db", f"{SAVE_DIR}/medical_reports.db")
    print("Database saved")

# =========================
# SAVE FAISS INDEX
# =========================
if 'index' in globals():
    faiss.write_index(index, f"{SAVE_DIR}/faiss_medical.index")
    print("FAISS index saved")

# =========================
# SAVE PICKLE TEXT DATA
# =========================
if os.path.exists("faiss_texts.pkl"):
    shutil.copy("faiss_texts.pkl", f"{SAVE_DIR}/faiss_texts.pkl")
    print("Pickle file saved")

# =========================
# SAVE SENTENCE EMBEDDER
# =========================
if 'embedder' in globals():
    embedder.save(f"{SAVE_DIR}/sentence_embedder")
    print("SentenceTransformer saved")

# =========================
# SAVE FULL MODEL (MERGED + TOKENIZER) ✅ CRITICAL
# =========================
FINAL_MODEL_PATH = f"{SAVE_DIR}/final_merged_model"

if 'model' in globals() and 'tokenizer' in globals():
    print("Merging LoRA with base model...")

    try:
        # Merge LoRA into base model
        merged_model = model.merge_and_unload()

        # Save FULL standalone model
        merged_model.save_pretrained(FINAL_MODEL_PATH, safe_serialization=True)

        # Save tokenizer WITH model (fixes tokenizer errors forever)
        tokenizer.save_pretrained(FINAL_MODEL_PATH)

        print("✅ Full merged model + tokenizer saved")

    except Exception as e:
        print("❌ Merge failed, saving fallback:", e)

        # Fallback: still save something usable
        model.save_pretrained(f"{SAVE_DIR}/fine_tuned_model")
        tokenizer.save_pretrained(f"{SAVE_DIR}/fine_tuned_model")

        print("⚠️ Saved LoRA model (not merged)")

# =========================
# SAVE SPACY MODEL INFO
# =========================
with open(f"{SAVE_DIR}/spacy_info.json", "w") as f:
    json.dump({"model_name": "en_core_web_sm"}, f)

# =========================
# SAVE IMPORTANT VARIABLES
# =========================
checkpoint_data = {}

if 'embeddings' in globals():
    checkpoint_data['embeddings'] = embeddings

if 'MEDICAL_PATTERNS' in globals():
    checkpoint_data['MEDICAL_PATTERNS'] = MEDICAL_PATTERNS

joblib.dump(checkpoint_data, f"{SAVE_DIR}/runtime_variables.pkl")

# =========================
# SAVE APP FILE
# =========================
if os.path.exists("app.py"):
    shutil.copy("app.py", f"{SAVE_DIR}/app.py")
    print("Streamlit app saved")

# =========================
# SAVE MODEL CONFIG (IMPORTANT)
# =========================
with open(f"{SAVE_DIR}/model_config.json", "w") as f:
    json.dump({
        "base_model": "google/gemma-2b-it",
        "type": "merged_full_model"
    }, f)

# =========================
# SAVE REQUIREMENTS (FUTURE SAFE)
# =========================
with open(f"{SAVE_DIR}/requirements.txt", "w") as f:
    f.write("""transformers
peft
accelerate
bitsandbytes
sentencepiece
faiss-cpu
sentence-transformers
streamlit
spacy
nltk
datasets
trl
joblib
""")


print("\n==============================")
print("✅ EVERYTHING SAVED SUCCESSFULLY")
print("💾 Model is now FULLY standalone")
print("🚀 No tokenizer / LoRA / path errors ever again")
print("==============================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving everything to: /content/drive/MyDrive/genai_project_full_backup
DataFrame saved
Database saved
FAISS index saved
Pickle file saved


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SentenceTransformer saved
Merging LoRA with base model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Full merged model + tokenizer saved

✅ EVERYTHING SAVED SUCCESSFULLY
💾 Model is now FULLY standalone
🚀 No tokenizer / LoRA / path errors ever again


In [ ]:
# =========================
# RELOAD FULL PROJECT (FIXED)
# =========================

# Install dependencies
!pip install -q transformers peft accelerate bitsandbytes
!pip install -q faiss-cpu sentence-transformers
!pip install -q streamlit pyngrok
!pip install -q spacy rouge-score nltk datasets trl joblib
!python -m spacy download en_core_web_sm -q

import os
import json
import pickle
import sqlite3
import joblib
import torch
import pandas as pd
import numpy as np
import faiss
import nltk
import spacy

from google.colab import drive
drive.mount('/content/drive')

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# PATH
# =========================
SAVE_DIR = "/content/drive/MyDrive/genai_project_full_backup"
MODEL_PATH = f"{SAVE_DIR}/final_merged_model"

print("Loading from:", SAVE_DIR)

# =========================
# LOAD DATAFRAME
# =========================
df = pd.read_csv(f"{SAVE_DIR}/medical_reports_processed.csv")
print("DataFrame loaded")

# =========================
# LOAD DATABASE
# =========================
conn = sqlite3.connect(f"{SAVE_DIR}/medical_reports.db")
print("Database connected")

# =========================
# LOAD FAISS INDEX
# =========================
index = faiss.read_index(f"{SAVE_DIR}/faiss_medical.index")
print("FAISS index loaded")

# =========================
# LOAD PICKLE FILE
# =========================
with open(f"{SAVE_DIR}/faiss_texts.pkl", "rb") as f:
    faiss_data = pickle.load(f)

print("Pickle text data loaded")

# =========================
# LOAD SENTENCE EMBEDDER
# =========================
embedder = SentenceTransformer(f"{SAVE_DIR}/sentence_embedder")
print("SentenceTransformer loaded")

# =========================
# LOAD MODEL + TOKENIZER (CRITICAL FIX)
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=False,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

# Fix padding (important)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model.eval()

print("Merged model + tokenizer loaded successfully")

# =========================
# LOAD VARIABLES
# =========================
runtime_vars = joblib.load(f"{SAVE_DIR}/runtime_variables.pkl")

if 'embeddings' in runtime_vars:
    embeddings = runtime_vars['embeddings']

if 'MEDICAL_PATTERNS' in runtime_vars:
    MEDICAL_PATTERNS = runtime_vars['MEDICAL_PATTERNS']

print("Runtime variables loaded")

# =========================
# LOAD NLP
# =========================
nlp = spacy.load("en_core_web_sm")

# =========================
# NLTK
# =========================
nltk.download('punkt', quiet=True)

print("\nEVERYTHING RELOADED SUCCESSFULLY")
print("Project ready to use 🚀")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 101.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart run

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer loaded


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Merged model + tokenizer loaded successfully
Runtime variables loaded

EVERYTHING RELOADED SUCCESSFULLY
Project ready to use 🚀


# Dashboard

In [ ]:
# ── Upload app.py from your computer ─────────────────────────────────────────
from google.colab import files
print("📂 Upload your app.py file...")
uploaded = files.upload()   # select the app.py you saved in Notepad

# Verify it uploaded correctly
import os
if os.path.exists('app.py'):
    size = os.path.getsize('app.py')
    print(f"✅ app.py uploaded successfully ({size} bytes)")
else:
    print("❌ Upload failed — try again")

📂 Upload your app.py file...


Saving app.py to app.py
✅ app.py uploaded successfully (29604 bytes)


In [ ]:
!pip install -q sentencepiece tiktoken

In [ ]:
from pyngrok import ngrok
import subprocess, time, os

# Optional: paste your free token from ngrok.com/signup
ngrok.set_auth_token("33enCaDqKoK8pMHbl73PYBRd1uX_3a1tzAj6E8fbymZdqBANm")

# Kill any old streamlit process
os.system('pkill -f streamlit 2>/dev/null')
time.sleep(2)

# Start Streamlit
proc = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--server.fileWatcherType', 'none'
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# Open public tunnel
public_url = ngrok.connect(8501)
print("=" * 55)
print(f"  🌐 OPEN THIS LINK: {public_url}")
print("=" * 55)
print("  First load takes ~60s (loading Gemma weights)")
print("  Use RAG Only mode for instant results")
print("  Run ngrok.kill() to stop")

  🌐 OPEN THIS LINK: NgrokTunnel: "https://unalloyed-eddy-cactuslike.ngrok-free.dev" -> "http://localhost:8501"
  First load takes ~60s (loading Gemma weights)
  Use RAG Only mode for instant results
  Run ngrok.kill() to stop


In [ ]:
ngrok.kill()

# Comparison with Research Models

In [ ]:
import gc, torch, json, re, warnings
import numpy as np
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
)
from peft import PeftModel
warnings.filterwarnings('ignore')

# Install missing dependency for BioGptTokenizer
!pip install -q sacremoses

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Free all GPU memory
# ─────────────────────────────────────────────────────────────────────────────
print('🧹 Freeing GPU memory...')
for name in ['model', 'ft_model', 'ft_base', 'base_model',
             'raw_model', 'biomistral_model', 'meditron_model',
             'biomedlm_model', 'biogpt_model']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f'✅ GPU cleared — Free: {free:.2f}GB / Total: {total:.2f}GB\n')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Setup
# ─────────────────────────────────────────────────────────────────────────────
scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method4
train_size = int(0.85 * len(df)) # Define train_size here
eval_df  = df.iloc[train_size:].reset_index(drop=True).head(20)

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Prompt builders
# ─────────────────────────────────────────────────────────────────────────────

def build_biogpt_prompt(report_text):
    """
    BioGPT — Microsoft Research 2022
    Paper: 'BioGPT: Generative Pre-trained Transformer for
            Biomedical Text Generation and Mining'
    Completion style — trained on 15M PubMed abstracts
    """
    return (
        f"The following laboratory results were obtained: "
        f"{report_text.strip()} "
        f"These results indicate that"
    )

def build_biomedlm_prompt(report_text):
    """
    BioMedLM — Stanford CRFM 2022, cited heavily in 2024-2025
    Paper: 'BioMedLM: A 2.7B Parameter Language Model
            Trained On Biomedical Text'
    Completion style — trained on PubMed + PubMed Central
    """
    return (
        f"Medical Report: {report_text.strip()}\n"
        f"Plain Language Explanation:"
    )

def build_finetuned_prompt(report_text):
    """
    Our model — Gemma-2B-IT + LoRA fine-tuned on medical dataset
    Uses exact chat format from training
    """
    return (
        "<start_of_turn>user\n"
        "Explain the following medical report in simple plain English. "
        "Write 2-3 clear sentences. Do not repeat words.\n\n"
        f"{report_text.strip()}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Stopping criteria
# ─────────────────────────────────────────────────────────────────────────────

def make_stop_criteria(tok, extra_strings=None):
    stop_ids = {tok.eos_token_id}
    if extra_strings:
        for s in extra_strings:
            tid = tok.convert_tokens_to_ids(s)
            if tid and tid != getattr(tok, 'unk_token_id', None):
                stop_ids.add(tid)
    stop_ids = list(stop_ids)

    class StopOnTokens(StoppingCriteria):
        def __call__(self, input_ids, scores, **kwargs):
            return input_ids[0][-1].item() in stop_ids

    return StoppingCriteriaList([StopOnTokens()])

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Output cleaner
# ─────────────────────────────────────────────────────────────────────────────

def clean_output(text):
    if not text:
        return ''
    # Remove special tags
    for tag in ['<end_of_turn>', '<start_of_turn>', 'model\n',
                'user\n', 'assistant\n', '[INST]', '[/INST]']:
        text = text.replace(tag, '')
    # Remove HTML junk
    text = re.sub(r'<[^>]{0,40}>', '', text)
    # Collapse repeated words: "Sever Sever Sever" → "Sever"
    text = re.sub(r'\b(\w+)(\s+\1){2,}\b', r'\1', text, flags=re.IGNORECASE)
    # Collapse no-space stutter: "severseversever" → "sever"
    text = re.sub(r'\b(\w{4,}?)\1{2,}\b', r'\1', text, flags=re.IGNORECASE)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Deduplicate sentences, keep max 3
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text)
                 if len(s.strip()) > 5]
    seen, unique = [], []
    for s in sentences:
        key = s.lower().strip('.!? ')
        if key not in seen:
            seen.append(key)
            unique.append(s)
    result = ' '.join(unique[:3])
    # Reject if still garbage
    if len(set(result.lower().split())) < 4:
        return ''
    return result

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Generate function
# ─────────────────────────────────────────────────────────────────────────────

def generate(mdl, tok, prompt, use_greedy=False,
             max_new=150, extra_stop=None):
    mdl.config.use_cache = True
    encoded = tok(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=450,
        add_special_tokens=True,
        padding=False
    )
    input_ids      = encoded['input_ids'].to(mdl.device)
    attention_mask = torch.ones_like(input_ids)
    input_len      = input_ids.shape[1]
    eos_id         = tok.eos_token_id

    if use_greedy:
        # Beam search — stable, no repetition, best for fine-tuned model
        gen_kwargs = dict(
            input_ids            = input_ids,
            attention_mask       = attention_mask,
            max_new_tokens       = max_new,
            do_sample            = False,
            num_beams            = 4,
            repetition_penalty   = 2.0,
            no_repeat_ngram_size = 4,
            early_stopping       = True,
            pad_token_id         = eos_id,
            eos_token_id         = eos_id,
            use_cache            = True,
            stopping_criteria    = make_stop_criteria(tok, extra_stop)
        )
    else:
        # Sampling — for baseline models
        gen_kwargs = dict(
            input_ids            = input_ids,
            attention_mask       = attention_mask,
            max_new_tokens       = max_new,
            do_sample            = True,
            temperature          = 0.7,
            top_p                = 0.90,
            top_k                = 40,
            repetition_penalty   = 1.8,
            no_repeat_ngram_size = 4,
            pad_token_id         = eos_id,
            eos_token_id         = eos_id,
            use_cache            = True,
            stopping_criteria    = make_stop_criteria(tok, extra_stop)
        )

    with torch.no_grad():
        output_ids = mdl.generate(**gen_kwargs)

    new_tokens = output_ids[0][input_len:]
    decoded    = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return clean_output(decoded)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Evaluate function
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model(mdl, tok, label, prompt_fn,
                   use_greedy=False, extra_stop=None):
    r1s, r2s, rLs, bleus = [], [], [], []
    skipped = 0

    for i, (_, row) in enumerate(eval_df.iterrows()):
        prompt = prompt_fn(row['report_clean'])
        pred   = generate(mdl, tok, prompt,
                          use_greedy=use_greedy,
                          extra_stop=extra_stop)
        ref    = row['explanation_clean']

        if len(pred.split()) < 4:
            skipped += 1
            continue

        # Print first 2 samples
        if i < 2:
            print(f'    Sample {i+1}:')
            print(f'    REF  : {ref[:85]}')
            print(f'    PRED : {pred[:85]}\n')

        s = scorer.score(ref, pred)
        r1s.append(s['rouge1'].fmeasure)
        r2s.append(s['rouge2'].fmeasure)
        rLs.append(s['rougeL'].fmeasure)
        bleus.append(sentence_bleu(
            [ref.split()], pred.split(), smoothing_function=smoother
        ))

    if not r1s:
        print(f'  ❌ All outputs invalid for: {label}')
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'bleu': 0.0}

    result = {
        'rouge1': float(np.mean(r1s)),
        'rouge2': float(np.mean(r2s)),
        'rougeL': float(np.mean(rLs)),
        'bleu':   float(np.mean(bleus))
    }
    print(f'\n  {"═"*50}')
    print(f'  {label}')
    print(f'  {"═"*50}')
    print(f'  ROUGE-1 : {result["rouge1"]:.4f}')
    print(f'  ROUGE-2 : {result["rouge2"]:.4f}')
    print(f'  ROUGE-L : {result["rougeL"]:.4f}')
    print(f'  BLEU    : {result["bleu"]:.4f}')
    print(f'  Valid   : {len(r1s)}/20  (skipped {skipped})')
    print(f'  {"═"*50}\n')
    return result

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — Load and evaluate each model ONE AT A TIME
#           Free GPU between each to avoid OOM
# ─────────────────────────────────────────────────────────────────────────────

all_scores = {}

# Load saved baseline + finetuned scores from previous cell
try:
    with open('eval_scores.json') as f:
        saved = json.load(f)
    all_scores['baseline'] = saved['baseline']
    all_scores['finetuned'] = saved['finetuned']
    print(f'✅ Loaded saved scores from eval_scores.json')
    print(f'   Baseline ROUGE-1  : {all_scores["baseline"]["rouge1"]:.4f}')
    print(f'   Fine-tuned ROUGE-1: {all_scores["finetuned"]["rouge1"]:.4f}\n')
except:
    print('⚠️  eval_scores.json not found — run baseline/finetuned cell first\n')
    all_scores['baseline']  = {'rouge1':0.0,'rouge2':0.0,'rougeL':0.0,'bleu':0.0}
    all_scores['finetuned'] = {'rouge1':0.0,'rouge2':0.0,'rougeL':0.0,'bleu':0.0}

# ── RESEARCH MODEL 1: BioGPT ─────────────────────────────────────────────────
print('='*55)
print('📊 [1/2] Research Model 1 — BioGPT (Microsoft, 2022)')
print('   Paper : BioGPT: Generative Pre-trained Transformer')
print('           for Biomedical Text Generation and Mining')
print('   Venue : Briefings in Bioinformatics, 2022')
print('   Data  : 15 million PubMed abstracts')
print('   Params: 347M — very lightweight')
print('='*55)

biogpt_tok = AutoTokenizer.from_pretrained('microsoft/biogpt')
if biogpt_tok.pad_token is None:
    biogpt_tok.pad_token = biogpt_tok.eos_token
biogpt_tok.padding_side = 'right'

biogpt_mdl = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    torch_dtype=torch.float16,
    device_map='auto'
)
biogpt_mdl.config.use_cache = True
if hasattr(biogpt_mdl, 'gradient_checkpointing_disable'):
    biogpt_mdl.gradient_checkpointing_disable()
biogpt_mdl.eval()

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f'✅ BioGPT loaded — GPU free: {free:.2f}GB\n')

all_scores['biogpt'] = evaluate_model(
    biogpt_mdl, biogpt_tok,
    'BioGPT — Microsoft Research (Briefings Bioinformatics 2022)',
    build_biogpt_prompt,
    use_greedy=False
)

# Free BioGPT from GPU
del biogpt_mdl
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1024**3
print(f'🧹 BioGPT unloaded — GPU free: {free:.2f}GB\n')

# ── RESEARCH MODEL 2: BioMedLM ───────────────────────────────────────────────
print('='*55)
print('📊 [2/2] Research Model 2 — BioMedLM-2.7B (Stanford, 2022)')
print('   Paper : BioMedLM: A 2.7B Parameter Language Model')
print('           Trained On Biomedical Text')
print('   Venue : Stanford CRFM 2022, cited in 2024-2025 benchmarks')
print('   Data  : PubMed + PubMed Central (~4.5B tokens)')
print('   Params: 2.7B in 4-bit = ~2GB')
print('='*55)

biomedlm_tok = AutoTokenizer.from_pretrained(
    'stanford-crfm/BioMedLM',
    trust_remote_code=True
)
if biomedlm_tok.pad_token is None:
    biomedlm_tok.pad_token = biomedlm_tok.eos_token
biomedlm_tok.padding_side = 'right'

biomedlm_mdl = AutoModelForCausalLM.from_pretrained(
    'stanford-crfm/BioMedLM',
    quantization_config=bnb_4bit,
    device_map='auto',
    trust_remote_code=True
)
biomedlm_mdl.config.use_cache = True
if hasattr(biomedlm_mdl, 'gradient_checkpointing_disable'):
    biomedlm_mdl.gradient_checkpointing_disable()
biomedlm_mdl.eval()

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f'✅ BioMedLM loaded — GPU free: {free:.2f}GB\n')

all_scores['biomedlm'] = evaluate_model(
    biomedlm_mdl, biomedlm_tok,
    'BioMedLM-2.7B — Stanford CRFM (2022, cited 2024-2025)',
    build_biomedlm_prompt,
    use_greedy=False
)

# Free BioMedLM from GPU
del biomedlm_mdl
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1024**3
print(f'🧹 BioMedLM unloaded — GPU free: {free:.2f}GB\n')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9 — Save all scores
# ─────────────────────────────────────────────────────────────────────────────
with open('eval_scores_research.json', 'w') as f:
    json.dump(all_scores, f, indent=2)
print('✅ eval_scores_research.json saved!')

try:
    import shutil
    shutil.copy('eval_scores_research.json',
                f'{DRIVE_BASE}/eval_scores_research.json')
    print('✅ Synced to Google Drive\n')
except:
    pass

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — Final comparison table
# ─────────────────────────────────────────────────────────────────────────────
models_display = [
    ('baseline',  'Raw Gemma-2B (Baseline)         '),
    ('biogpt',    'BioGPT-347M (Microsoft 2022)     '),
    ('biomedlm',  'BioMedLM-2.7B (Stanford 2022)   '),
    ('finetuned', 'Gemma-2B-IT + LoRA FT (Ours)    '),
]

print('\n' + '═'*70)
print(f'  {"Model":<36} {"R-1":>7} {"R-2":>7} {"R-L":>7} {"BLEU":>7}')
print('─'*70)
for key, name in models_display:
    s = all_scores.get(key, {})
    r1 = s.get('rouge1', 0.0)
    r2 = s.get('rouge2', 0.0)
    rl = s.get('rougeL', 0.0)
    bl = s.get('bleu',   0.0)
    print(f'  {name} {r1:>7.4f} {r2:>7.4f} {rl:>7.4f} {bl:>7.4f}')
print('═'*70)

# Gain of our model vs research models
print('\n📈 Our Model Gain vs Research Models:')
our = all_scores.get('finetuned', {})
for key, name in [('biogpt',   'BioGPT  '),
                  ('biomedlm', 'BioMedLM')]:
    them = all_scores.get(key, {})
    for m, lbl in [('rouge1','ROUGE-1'), ('bleu','BLEU')]:
        gain  = (our.get(m, 0) - them.get(m, 0)) * 100
        arrow = '📈' if gain > 0 else '📉'
        print(f'  {arrow}  vs {name}  {lbl}: {gain:+.1f}%')

print('\n✅ Research comparison complete!')
print('   Update app.py Tab 2 to load eval_scores_research.json for the chart.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 42.9 MB/s eta 0:00:00
🧹 Freeing GPU memory...
✅ GPU cleared — Free: 14.34GB / Total: 14.56GB

⚠️  eval_scores.json not found — run baseline/finetuned cell first

📊 [1/2] Research Model 1 — BioGPT (Microsoft, 2022)
   Paper : BioGPT: Generative Pre-trained Transformer
           for Biomedical Text Generation and Mining
   Venue : Briefings in Bioinformatics, 2022
   Data  : 15 million PubMed abstracts
   Params: 347M — very lightweight


pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: stanford-crfm/BioMedLM
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...31}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...31}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BioMedLM loaded — GPU free: 5.37GB

    Sample 1:
    REF  : Severely elevated glucose; evaluate for diabetic ketoacidosis or hyperosmolar state.
    PRED : Caring for the patient with diabetes at home during and after stroke. The authors des

    Sample 2:
    REF  : Severely elevated glucose; evaluate for diabetic ketoacidosis or hyperosmolar state.
    PRED : The effect of early life stress on cognitive performance in the emergency department 


  ══════════════════════════════════════════════════
  BioMedLM-2.7B — Stanford CRFM (2022, cited 2024-2025)
  ══════════════════════════════════════════════════
  ROUGE-1 : 0.0274
  ROUGE-2 : 0.0009
  ROUGE-L : 0.0274
  BLEU    : 0.0037
  Valid   : 20/20  (skipped 0)
  ══════════════════════════════════════════════════

🧹 BioMedLM unloaded — GPU free: 14.31GB

✅ eval_scores_research.json saved!

══════════════════════════════════════════════════════════════════════
  Model                                    R-1     R-2     R-L    BLEU
───